# AI TRPG text_model manual test

This notebook tests all currently adapted text models:
- `deepseek`
- `chatgpt`
- `gemini`

It does not require editing `data_UserPreference.json` by hand.


In [13]:
import json
import os
from pathlib import Path

from openai import OpenAI
from pydantic import BaseModel

from text_model.function_TextModel import (
    collect_stream_text,
    get_mini_reply,
    get_normal_reply,
    get_reply,
    get_selected_text_model_config,
    get_stream_reply,
    get_structured_reply,
)


In [14]:
MODEL_CODES = ["deepseek", "chatgpt", "gemini"]
TMP_PREF_DIR = Path("data") / "_tmp_notebook_prefs"
TMP_PREF_DIR.mkdir(parents=True, exist_ok=True)

messages = [
    {"role": "system", "content": "Reply in concise Chinese."},
    {"role": "user", "content": "请用一句话介绍你自己。"},
]

def make_pref_file(model_code: str) -> Path:
    pref_path = TMP_PREF_DIR / f"pref_{model_code}.json"
    pref_data = {
        "language": "zh-CN",
        "text_model": {"code": model_code},
    }
    pref_path.write_text(json.dumps(pref_data, ensure_ascii=False, indent=2), encoding="utf-8")
    return pref_path

def show_model_info(model_code: str):
    pref_path = make_pref_file(model_code)
    cfg = get_selected_text_model_config(preference_path=pref_path)
    print(f"model_code = {model_code}")
    print("provider =", cfg.dependence)
    print("base_model =", cfg.base_model)
    print("base_url =", cfg.base_url)
    print("mini_supported =", cfg.features["mini_version"].supported)
    print()

def run_and_print(title: str, fn):
    print(title)
    try:
        result = fn()
        print(result)
    except Exception as exc:
        print(type(exc).__name__, exc)
    print()

messages


[{'role': 'system', 'content': 'Reply in concise Chinese.'},
 {'role': 'user', 'content': '请用一句话介绍你自己。'}]

## Model info check

Run this first to verify model selection and feature flags.


In [15]:
for code in MODEL_CODES:
    show_model_info(code)


model_code = deepseek
provider = OpenAI
base_model = deepseek-chat
base_url = https://api.deepseek.com
mini_supported = False

model_code = chatgpt
provider = OpenAI
base_model = gpt-5.2
base_url = None
mini_supported = True

model_code = gemini
provider = Google
base_model = gemini-2.5-pro
base_url = https://generativelanguage.googleapis.com
mini_supported = True



## Gemini model discovery

If Gemini returns 404, run this cell first and use the exact model id returned here in `data_TextModel.yml`.


In [16]:
client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

for model in client.models.list():
    model_id = getattr(model, "id", "")
    if "gemini" in model_id.lower():
        print(model_id)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/gemini-embedding-001
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025


## Structured outputs

This path currently targets native OpenAI Responses API models, so use `chatgpt` here.


In [25]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


In [ ]:
structured_messages = [
    {
        "role": "system",
        "content": (
            "Extract one calendar event. "
            "Field rules: "
            "`name` is the event title, not a person's name or item name. "
            "`date` is the event date. "
            "`participants` is a list of people involved in the event."
        ),
    },
    {
        "role": "user",
        "content": "Alice and Bob are going to a science fair on Friday.",
    },
]


chatgpt_pref_path = make_pref_file("chatgpt")
event = get_structured_reply(
    input=structured_messages,
    output_schema=CalendarEvent,
    preference_path=chatgpt_pref_path,
    timeout=60,
)

print(event)
print(event.model_dump())


## Real request tests

Before running the cells below, make sure your `.env` contains valid keys:
- `DEEPSEEK_API_KEY`
- `OPENAI_API_KEY`
- `GEMINI_API_KEY`

These cells continue on errors so you can compare all three providers in one run.


In [19]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    run_and_print(
        f"===== {code} | normal =====",
        lambda pref_path=pref_path: get_normal_reply(input=messages, preference_path=pref_path, timeout=60),
    )


===== deepseek | normal =====
我是DeepSeek，由深度求索公司创造的AI助手，致力于用热情与智慧为你提供帮助！😊

===== chatgpt | normal =====
我是一个基于大模型的AI助手，能够用中文为你提供信息查询、写作润色、代码与学习辅导等帮助。

===== gemini | normal =====
我是一个由谷歌训练的大型语言模型。



In [20]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    run_and_print(
        f"===== {code} | get_reply non-stream =====",
        lambda pref_path=pref_path: get_reply(is_stream=False, input=messages, preference_path=pref_path, timeout=60),
    )


===== deepseek | get_reply non-stream =====
我是DeepSeek，由深度求索公司创造的AI助手，致力于用热情与智慧为你提供帮助。

===== chatgpt | get_reply non-stream =====
我是一个能用中文简洁回答问题、协助写作与编程并提供信息整理的AI助手。

===== gemini | get_reply non-stream =====
我是一个由谷歌训练的大型语言模型。



In [21]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    print(f"===== {code} | stream =====")
    try:
        chunks = []
        for chunk in get_stream_reply(input=messages, preference_path=pref_path, timeout=60):
            print(chunk, end="")
            chunks.append(chunk)
        print("\n")
        print("joined =", collect_stream_text(chunks))
    except Exception as exc:
        print(type(exc).__name__, exc)
    print()


===== deepseek | stream =====
我是DeepSeek，由深度求索公司创造的AI助手，致力于用热情与智慧为你提供帮助。

joined = 我是DeepSeek，由深度求索公司创造的AI助手，致力于用热情与智慧为你提供帮助。

===== chatgpt | stream =====
我是一个可以用中文为你快速解答问题、提供建议并协助写作与编程的AI助手。

joined = 我是一个可以用中文为你快速解答问题、提供建议并协助写作与编程的AI助手。

===== gemini | stream =====
我是一个由谷歌训练的大型语言模型。

joined = 我是一个由谷歌训练的大型语言模型。



In [22]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    print(f"===== {code} | get_reply stream =====")
    try:
        chunks = []
        for chunk in get_reply(is_stream=True, input=messages, preference_path=pref_path, timeout=60):
            print(chunk, end="")
            chunks.append(chunk)
        print("\n")
        print("joined =", collect_stream_text(chunks))
    except Exception as exc:
        print(type(exc).__name__, exc)
    print()


===== deepseek | get_reply stream =====
我是DeepSeek，由深度求索公司创造的AI助手，致力于用热情与智能为您提供帮助。

joined = 我是DeepSeek，由深度求索公司创造的AI助手，致力于用热情与智能为您提供帮助。

===== chatgpt | get_reply stream =====
我是一个基于大规模语言模型的AI助手，能用中文为你提供问答、写作、翻译与分析等帮助。

joined = 我是一个基于大规模语言模型的AI助手，能用中文为你提供问答、写作、翻译与分析等帮助。

===== gemini | get_reply stream =====
我是一个由谷歌训练的大型语言模型。

joined = 我是一个由谷歌训练的大型语言模型。



In [23]:
for code in MODEL_CODES:
    pref_path = make_pref_file(code)
    run_and_print(
        f"===== {code} | mini =====",
        lambda pref_path=pref_path: get_mini_reply(input=messages, preference_path=pref_path, timeout=60),
    )


===== deepseek | mini =====
ValueError Feature 'mini_version' not supported for model 'deepseek'

===== chatgpt | mini =====
我是由 OpenAI 训练的语言模型助手，致力于回答问题、提供信息并协助完成各类任务。

===== gemini | mini =====
我是一个大型语言模型，由OpenAI训练。



In [24]:
for path in TMP_PREF_DIR.glob("pref_*.json"):
    path.unlink(missing_ok=True)

print("temporary preference files cleaned")


temporary preference files cleaned
